# Clinical Guidelines RAG Assistant
### Week 5 Exercise — Stella Oiro (Andela AI Engineering Bootcamp)

A RAG-powered chatbot that answers clinical questions from a structured medical knowledge base.

**Knowledge base covers:**
- Clinical guidelines: hypertension, type 2 diabetes, asthma, heart failure, sepsis
- Drug references: antibiotics, antihypertensives, insulin types
- Scoring tools: Wells score (PE/DVT), CURB-65, CHA2DS2-VASc, GCS

**Pipeline:**
1. Load markdown files with LangChain `DirectoryLoader`
2. Chunk with `RecursiveCharacterTextSplitter`
3. Embed with HuggingFace `all-MiniLM-L6-v2`
4. Store in ChromaDB
5. Retrieve + generate answers with GPT-4.1-mini
6. Gradio `ChatInterface` for the UI

In [ ]:
# Imports

import os
import glob
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

import gradio as gr

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-'):
    print(f"OpenAI API key found: {api_key[:12]}...")
else:
    print("OpenAI API key not found — check your .env file")

In [ ]:
# Constants

MODEL    = "gpt-4.1-mini"
DB_NAME  = "clinical_vector_db"
KB_PATH  = "clinical-knowledge-base"

print(f"Model: {MODEL}")
print(f"Knowledge base: {KB_PATH}")
print(f"Vector store: {DB_NAME}")

## Step 1 — Load the clinical knowledge base

The knowledge base is organised into three sub-folders:
- `guidelines/` — condition-specific clinical guidelines
- `drugs/` — drug reference sheets
- `scoring-tools/` — clinical scoring systems

We attach a `doc_type` metadata tag to each document so we can filter or colour-code results later.

In [ ]:
# Load all markdown files from each sub-folder

def load_knowledge_base(path):
    folders   = glob.glob(f"{path}/*")
    documents = []

    for folder in folders:
        doc_type = os.path.basename(folder)
        loader   = DirectoryLoader(
            folder,
            glob="**/*.md",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}
        )
        docs = loader.load()
        for doc in docs:
            doc.metadata["doc_type"] = doc_type
        documents.extend(docs)
        print(f"  {doc_type}: {len(docs)} file(s) loaded")

    return documents


print("Loading knowledge base...")
documents = load_knowledge_base(KB_PATH)
print(f"\nTotal documents: {len(documents)}")
print(f"Doc types: {set(d.metadata['doc_type'] for d in documents)}")

## Step 2 — Chunk the documents

`RecursiveCharacterTextSplitter` splits on paragraph boundaries first, then sentences, then characters.
This keeps clinical tables and numbered lists intact where possible.

In [ ]:
# Chunk documents

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk (first):")
print(f"  Type: {chunks[0].metadata['doc_type']}")
print(f"  Source: {os.path.basename(chunks[0].metadata['source'])}")
print(f"  Length: {len(chunks[0].page_content)} chars")
print(f"  Preview: {chunks[0].page_content[:200]}...")

## Step 3 — Embed and store in ChromaDB

We use the `all-MiniLM-L6-v2` HuggingFace model (384-dimensional embeddings, runs locally — no API cost).  
ChromaDB persists to disk so we only embed once.

In [ ]:
# Create embeddings and vector store

print("Loading HuggingFace embedding model (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Delete existing collection so we always rebuild from current knowledge base
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Deleted existing vector store at '{DB_NAME}'")

print("Building ChromaDB vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

count      = vectorstore._collection.count()
sample_emb = vectorstore._collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dims       = len(sample_emb)

print(f"\nVector store ready:")
print(f"  Vectors: {count:,}")
print(f"  Dimensions: {dims:,}")

## Step 4 — Build the RAG chain

Two LangChain objects:
- **Retriever** — fetches the top-k most relevant chunks by cosine similarity
- **LLM** — generates the final answer using the retrieved context

In [ ]:
# Set up retriever and LLM

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

llm = ChatOpenAI(temperature=0, model_name=MODEL)

print(f"Retriever: top-8 chunks by cosine similarity")
print(f"LLM: {MODEL} (temperature=0 for factual answers)")

In [ ]:
# Test the retriever

test_query = "What is the first-line treatment for hypertension?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[{i}] {doc.metadata['doc_type']} / {os.path.basename(doc.metadata['source'])}")
    print(f"    {doc.page_content[:120]}...\n")

## Step 5 — Answer function

The system prompt tells the model it is a clinical assistant.  
It is instructed to:
- use the retrieved context to answer
- cite which guideline or document the answer comes from
- acknowledge uncertainty rather than hallucinate
- add a safety disclaimer for clinical decisions

In [ ]:
SYSTEM_PROMPT = """\
You are a clinical knowledge assistant for healthcare professionals.
You have access to excerpts from clinical guidelines, drug reference sheets, and scoring tools.

When answering:
- Use the provided context excerpts to give accurate, concise answers
- Mention the source guideline or reference (e.g. NICE NG136, ADA 2024) where available in the context
- If the context does not contain enough information, say so clearly — do not guess or invent clinical data
- Use UK/international units by default (mmol/L for glucose, mmHg for BP, etc.)
- Format responses with markdown: use tables, bullet points, and bold for clarity

Safety disclaimer: This tool is for reference only. All clinical decisions must be verified
with current local guidelines and made by a qualified clinician.

Context:
{context}
"""


def answer_question(question: str, history) -> str:
    """Retrieve relevant chunks and generate a clinical answer."""
    docs    = retriever.invoke(question)
    context = "\n\n---\n\n".join(
        f"[Source: {d.metadata['doc_type']} / {os.path.basename(d.metadata['source'])}]\n{d.page_content}"
        for d in docs
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])
    return response.content


# Quick test
print(answer_question("What is the CURB-65 score and when should I admit a patient?", []))

## Step 6 — Gradio Chat UI

In [ ]:
EXAMPLE_QUESTIONS = [
    "What are the blood pressure targets for a patient with diabetes?",
    "How do I calculate the Wells score for PE?",
    "What is the Sepsis Six and when should I use it?",
    "Which insulin should I use for a type 2 diabetic starting insulin?",
    "When should I anticoagulate a patient with atrial fibrillation?",
    "What antibiotic would you use for community-acquired pneumonia?",
    "What GCS score requires intubation?",
    "What are the four drug classes for heart failure with reduced ejection fraction?",
]

with gr.Blocks(title="Clinical Guidelines RAG Assistant", theme=gr.themes.Soft()) as ui:

    gr.Markdown("""
    # Clinical Guidelines RAG Assistant
    Ask clinical questions about guidelines, drug dosing, and scoring tools.
    Answers are grounded in a curated knowledge base of clinical reference documents.

    **Knowledge base:** Hypertension, T2DM, Asthma, Heart Failure, Sepsis,
    Antibiotics, Antihypertensives, Insulin, Wells score, CURB-65, CHA2DS2-VASc, GCS

    > This tool is for **reference only**. All clinical decisions must be verified
    > with current local guidelines and made by a qualified clinician.
    """)

    chatbot = gr.ChatInterface(
        fn=answer_question,
        type="messages",
        examples=EXAMPLE_QUESTIONS,
        chatbot=gr.Chatbot(
            height=520,
            show_label=False,
            render_markdown=True,
            type="messages"
        ),
        textbox=gr.Textbox(
            placeholder="Ask a clinical question...",
            show_label=False
        ),
    )

print("UI built.")

In [ ]:
ui.launch(inbrowser=True)